In [ ]:
from copy import deepcopy
from hmac import new
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def get_mnist_dataloaders(batch_size=64, train_size=50000, valid_size=10000, test_size=10000):
    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

    train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

    train_subset = Subset(train_dataset, range(train_size))
    valid_subset = Subset(train_dataset, range(train_size, train_size + valid_size))
    test_subset = Subset(test_dataset, range(test_size))

    train_data = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
    valid_data = DataLoader(valid_subset, batch_size=batch_size)
    test_data = DataLoader(test_subset, batch_size=batch_size)

    return train_data, valid_data, test_data

class ANN(nn.Module):
    def __init__(self, num_classes=10, input_size=(1, 28, 28)):
        super(ANN, self).__init__()
        self.input_size = input_size
        input_features = np.prod(self.input_size) if isinstance(self.input_size, tuple) else self.input_size
        self.fc1 = nn.Linear(input_features, 256)
        self.fc2 = nn.Linear(256, 512)
        self.fc3 = nn.Linear(512, 256)
        self.fc4 = nn.Linear(256, 128)
        self.fc5 = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = x.view(x.size(0), -1) if isinstance(self.input_size, tuple) else x
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = F.relu(self.fc3(x))
        x = self.dropout(x)
        x = F.relu(self.fc4(x))
        return self.fc5(x)

def loss_classifier(predictions, labels):
    loss_function = nn.CrossEntropyLoss()
    return loss_function(predictions, labels.view(-1))

def evaluate_model(model, dataloader, loss_f):
    model_val = deepcopy(model).to(device)
    model_val.eval()
    total_loss, correct, total_samples = 0.0, 0, 0

    with torch.no_grad():
        for features, labels in dataloader:
            features, labels = features.to(device), labels.to(device)
            predictions = model_val(features)
            batch_loss = loss_f(predictions, labels)
            total_loss += batch_loss.item() * features.size(0)  # weighted by batch size
            _, predicted = predictions.max(1)
            correct += predicted.eq(labels).sum().item()
            total_samples += labels.size(0)

    if total_samples == 0: return 0.0, 0.0

    avg_loss = total_loss / total_samples
    accuracy = correct / total_samples
    return avg_loss, accuracy

def train_step(model, optimizer, train_data, valid_data, loss_f):
    all_outputs, all_labels = [], []
    model.train()
    t1 = time.time()
    for tr_idx, (tr_features, tr_labels) in enumerate(train_data):
        #print(f"Shape of tr_features: {tr_features.shape}, Shape of tr_labels: {tr_labels.shape}")
        optimizer.zero_grad()
        tr_features, tr_labels = tr_features.to(device), tr_labels.to(device)
        tr_predictions = model(tr_features)
        tr_batch_loss = loss_f(tr_predictions, tr_labels)
        tr_batch_loss.backward()
        optimizer.step()

        all_outputs.append(tr_predictions.detach().cpu())
        all_labels.append(tr_labels.detach().cpu())
    ep_time = time.time() - t1

    tr_loss, tr_accuracy = evaluate_model(model, train_data, loss_f)
    val_loss, val_accuracy = evaluate_model(model, valid_data, loss_f)
    return [tr_loss, tr_accuracy, val_loss, val_accuracy], ep_time, torch.cat(all_outputs), torch.cat(all_labels)

def local_learning_redus(model, optimizer, train_data, test_data, epochs: int, loss_f, threshold):
    best_model = deepcopy(model).to(device)
    # best loss initialized as max possible loss
    best_loss = float('inf')
    history, training_times, evaling_times = [], [], []

    weights = torch.ones(len(train_data.dataset)) / len(train_data.dataset)
    if not threshold: 
        threshold = 1 / ( len(train_data.dataset))
    print(f"Threshold: {threshold}, Train: {len(train_data.dataset)}, Test: {len(test_data.dataset)}, Epochs: {epochs}")

    for e in range(epochs):
        ep = "0" + str(e + 1) if e + 1 < 10 else str(e + 1)

        selected_indices = (weights >= threshold).nonzero().squeeze()
        unselected_indices = (weights < threshold).nonzero().squeeze()

        selected_train_data = DataLoader(Subset(train_data.dataset, selected_indices), batch_size=train_data.batch_size)
        unselected_train_data = DataLoader(Subset(train_data.dataset, unselected_indices), batch_size=train_data.batch_size)

        selected_weights_np = weights[selected_indices].numpy()
        unselected_weights_np = weights[unselected_indices].numpy()

        selection_ratio = len(selected_indices) / len(weights)
        print(f"Epoch {ep} ------------------------------")
        print(f"- Selection Ratio: {selection_ratio:.4f}, Selected Samples: {len(selected_indices)}/{len(train_data.dataset)}")
        print(f"- Unselected Samples: {len(unselected_indices)}/{len(train_data.dataset)}")
        
        if selection_ratio == 0:
            print("No samples are left to train on")
            continue

        local_res, local_time, selected_outputs, selected_labels = train_step(model, optimizer, selected_train_data, test_data, loss_f)
        history.append(local_res)
        training_times.append(local_time)

        tr_loss, tr_acc, val_loss, val_acc = local_res
        if val_loss < best_loss:
            best_loss = val_loss
            best_model = deepcopy(model).to(device)
        print(f'- Training: Tr_Loss: {tr_loss:.4f}\t Tr_Acc: {tr_acc*100:.2f}%\t Val_Loss: {val_loss:.4f}\t Val_Acc: {val_acc*100:.2f}% \ttime: {local_time:.4f}s)')

        if selection_ratio != 1:    
            t1 = time.time() 
            unselected_outputs, unselected_labels = [], [] 
            
            for idx, (data, label) in enumerate(unselected_train_data) : 
                data, label = data.to(device), label.to(device) 
                unselected_outputs.append(model(data).detach().cpu()) 
                unselected_labels.append(label.detach().cpu() )
            unselected_outputs, unselected_labels = torch.cat(unselected_outputs), torch.cat(unselected_labels)

            t2 = time.time() 
            evaling_times.append(t2 - t1) 

            print(f"type of selected_outputs: {type(selected_outputs)}")
            print(f"type of unselected_outputs: {type(unselected_outputs)}")
            print(f"type of selected_labels: {type(selected_labels)}")
            print(f"type of unselected_labels: {type(unselected_labels)}")

            
            print(f"shape of selected_outputs: {selected_outputs.shape}")
            print(f"shape of unselected_outputs: {unselected_outputs.shape}")
            print(f"shape of selected_labels: {selected_labels.shape}")
            print(f"shape of unselected_labels: {unselected_labels.shape}")
            

            all_outputs = torch.cat((selected_outputs, unselected_outputs), dim=0)
            all_labels  = torch.cat((selected_labels,  unselected_labels),  dim=0)

        else : 
            all_outputs = selected_outputs
            all_labels = selected_labels
            evaling_times.append(0)

        errors = (all_outputs.argmax(axis=1) != all_labels)

        epsilon_t = (weights * errors).sum()
        alpha_t = 0.5 * np.log((1 - epsilon_t) / (epsilon_t + 1e-8)) # to avoid division by zero
        Z_t = 2 * np.sqrt(epsilon_t * (1 - epsilon_t))

        errors_tensor = torch.tensor(errors).to(weights.device)
        pos_factor = np.exp(alpha_t) / Z_t
        neg_factor = np.exp(-alpha_t) / Z_t

        weights *= torch.where(errors_tensor == 1, pos_factor, neg_factor)
        
        print(f"- Epsilon (Error): {epsilon_t:.4f}, Alpha (Weight): {alpha_t:.4f}, Z (Normalization): {Z_t:.4f}")
        print(f"- Max weights: {weights.max():.4f}, Min weights: {weights.min():.4f}, Distance of lowest to threshold: {weights.min() - threshold:.4f}")
        print(f"- No. of samples above threshold: {(weights >= threshold).sum()}/{len(weights)}, lower than threshold: {(weights < threshold).sum()}/{len(weights)}")

    return best_model, history, training_times

print("ANN model defined successfully.")
print(f"Device used: {device}")

ANN model defined successfully.
Device used: cpu


In [39]:
batch_size = 64
epochs = 50
train_size, valid_size, test_size = 10000, 2000, 0

thresholds = np.linspace(0, 1/2*train_size, 10)




print(f"Thresholds: {thresholds}")


train_data, valid_data, test_data = get_mnist_dataloaders(batch_size=batch_size, train_size=train_size,
                                                           valid_size=valid_size, test_size=test_size)
history = {}
for threshold in thresholds:
    print(f"Training with threshold: {threshold:.4f}")
    model = ANN(input_size=(1, 28, 28), num_classes=10).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_f = loss_classifier

    best_model, model_performance, training_times = local_learning_redus(model, optimizer, train_data, valid_data, epochs, loss_f, threshold)
    history[threshold] = [model_performance, training_times]


    print(f"Best model for threshold {threshold:.4f} trained successfully.")

Thresholds: [   0.          555.55555556 1111.11111111 1666.66666667 2222.22222222
 2777.77777778 3333.33333333 3888.88888889 4444.44444444 5000.        ]
Training with threshold: 0.0000
Threshold: 0.0001, Train: 10000, Test: 2000, Epochs: 50
Epoch 01 ------------------------------
- Selection Ratio: 1.0000, Selected Samples: 10000/10000
- Unselected Samples: 0/10000
- Training: Tr_Loss: 0.3376	 Tr_Acc: 89.53%	 Val_Loss: 0.3659	 Val_Acc: 88.65% 	time: 1.4699s)
- Epsilon (Error): 0.2992, Alpha (Weight): 0.4256, Z (Normalization): 0.9158
- Max weights: 0.0002, Min weights: 0.0001, Distance of lowest to threshold: -0.0000
- No. of samples above threshold: 2992/10000, lower than threshold: 7008/10000
Epoch 02 ------------------------------
- Selection Ratio: 0.2992, Selected Samples: 2992/10000
- Unselected Samples: 7008/10000


/var/folders/fy/1qq1n6j95tn7gy2bwgbsqlhc0000gn/T/ipykernel_44964/53839288.py:178: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  alpha_t = 0.5 * np.log((1 - epsilon_t) / (epsilon_t + 1e-8)) # to avoid division by zero
/var/folders/fy/1qq1n6j95tn7gy2bwgbsqlhc0000gn/T/ipykernel_44964/53839288.py:179: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  Z_t = 2 * np.sqrt(epsilon_t * (1 - epsilon_t))
/var/folders/fy/1qq1n6j95tn7gy2bwgbsqlhc0000gn/T/ipykernel_44964/53839288.py:181: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  errors_tensor = torch.tensor(errors).to(weights.device)
/var/folders/fy/1qq1n6j95tn7gy2bwgbsqlhc0000gn/T/ipykernel_44964/53839288.py:182: Deprec

- Training: Tr_Loss: 0.6571	 Tr_Acc: 80.55%	 Val_Loss: 0.4062	 Val_Acc: 91.10% 	time: 0.4043s)
type of selected_outputs: <class 'torch.Tensor'>
type of unselected_outputs: <class 'torch.Tensor'>
type of selected_labels: <class 'torch.Tensor'>
type of unselected_labels: <class 'torch.Tensor'>
shape of selected_outputs: torch.Size([2992, 10])
shape of unselected_outputs: torch.Size([7008, 10])
shape of selected_labels: torch.Size([2992])
shape of unselected_labels: torch.Size([7008])
- Epsilon (Error): 0.1419, Alpha (Weight): 0.8997, Z (Normalization): 0.6979
- Max weights: 0.0006, Min weights: 0.0000, Distance of lowest to threshold: -0.0001
- No. of samples above threshold: 1326/10000, lower than threshold: 8674/10000
Epoch 03 ------------------------------
- Selection Ratio: 0.1326, Selected Samples: 1326/10000
- Unselected Samples: 8674/10000
- Training: Tr_Loss: 0.2985	 Tr_Acc: 90.80%	 Val_Loss: 0.4085	 Val_Acc: 87.50% 	time: 0.1850s)
type of selected_outputs: <class 'torch.Tensor'>